In [ ]:
import copy
import random
import string

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# ================================
# 1. Setup
# ================================

BASE_SEED = 44

torch.manual_seed(BASE_SEED)
np.random.seed(BASE_SEED)

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"

print(f"Using device: {device}")

# Enable fast matmul on modern NVIDIA GPUs
if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

# ================================
# 2. Globals / Hyperparameters
# ================================

model_name = "gpt2"  # 124M params
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

prompt = "".join(random.choices(string.ascii_letters + string.digits, k=20))
N = 15  # number of completions
K = 15  # number of preference pairs to sample
epochs = 300  # DPO training epochs
NUM_RUNS = 10  # average over this many runs

# ================================
# 3. Data Generation
# ================================


def generate_completions(model, tokenizer, prompt, n=100, max_length=20, run_idx=0):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    completions = []

    batch_size = 1
    num_batches = n

    model.eval()
    with torch.no_grad():
        for _ in tqdm(
            range(num_batches),
            desc=f"Run {run_idx + 1}: Generating completions",
            leave=False,
        ):
            batch_inputs = {k: v.repeat(batch_size, 1) for k, v in inputs.items()}
            outputs = model.generate(
                **batch_inputs,
                max_length=max_length,
                do_sample=True,
                top_k=50,
                pad_token_id=tokenizer.eos_token_id,
            )
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            completions.extend(decoded)

    return completions


# ================================
# 4. Preference Oracle & Graph Estimation (Bradley–Terry)
# ================================


def sample_pairs(n_completions, n_pairs):
    """
    Uniformly sample unique unordered pairs (i, j) with i < j.
    """
    pairs = []
    existing = set()
    while len(pairs) < n_pairs:
        idx = np.random.choice(n_completions, 2, replace=False)
        idx = tuple(sorted(idx))
        if idx not in existing:
            existing.add(idx)
            pairs.append(idx)
    return pairs


def label_pairs(pairs, scores):
    """
    Given a list of pairs (i, j) and scores for each completion,
    label them so that the higher-scoring index is the winner.
    Returns list of (winner, loser) indices.
    """
    labeled_pairs = []
    for i, j in pairs:
        if scores[i] > scores[j]:
            labeled_pairs.append((i, j))  # i wins
        else:
            labeled_pairs.append((j, i))  # j wins
    return labeled_pairs


def fit_bradley_terry(n_completions, labeled_pairs, n_iters=1000, lr=0.01, l1_reg=0.01):
    """
    Fit a Bradley-Terry model to estimate per-completion scores
    from a subset of labeled preference pairs.
    """
    scores = torch.randn(n_completions, requires_grad=True)
    optimizer = torch.optim.Adam([scores], lr=lr)

    winners = torch.tensor([w for w, _ in labeled_pairs], dtype=torch.long)
    losers = torch.tensor([l for _, l in labeled_pairs], dtype=torch.long)

    for _ in range(n_iters):
        optimizer.zero_grad()
        diffs = scores[winners] - scores[losers]
        # Bradley–Terry likelihood (negative log-likelihood)
        loss = -torch.sum(torch.log(torch.sigmoid(diffs) + 1e-8))
        # Small L1 regularization for stability
        loss += l1_reg * torch.sum(torch.abs(scores))
        loss.backward()
        optimizer.step()

    scores_np = scores.detach().cpu().numpy()
    # Normalize scores for interpretability
    scores_np = (scores_np - scores_np.mean()) / (scores_np.std() + 1e-8)
    return scores_np


def construct_full_graph(scores):
    """
    Given scores for each completion (e.g., from Bradley–Terry),
    construct the full preference graph over all (i, j),
    returning list of (winner, loser) pairs.
    """
    n = len(scores)
    pairs = []
    for i in range(n - 1):
        for j in range(i + 1, n):
            if scores[i] > scores[j]:
                pairs.append((i, j))  # i wins
            else:
                pairs.append((j, i))  # j wins
    return pairs


# ================================
# 5. Fast batched log-probs
# ================================


def get_log_prob_sums_from_inputs(model, input_ids, attention_mask):
    """
    Compute sum of log p(tokens) per sequence for a batch.

    input_ids: [B, L]
    attention_mask: [B, L]
    Returns: tensor [B]
    """
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1, :]  # [B, L-1, V]
    labels = input_ids[:, 1:]  # [B, L-1]

    log_probs = nn.functional.log_softmax(logits, dim=-1)
    token_log_probs = log_probs.gather(dim=-1, index=labels.unsqueeze(-1)).squeeze(
        -1
    )  # [B, L-1]

    mask = attention_mask[:, 1:]  # [B, L-1]
    sum_log_probs = (token_log_probs * mask).sum(dim=-1)  # [B]
    return sum_log_probs


# ================================
# 6. Fast DPO Training Loop
# ================================


def train_dpo_epoch_save_log_probs_fast(
    policy_model,
    ref_log_probs_tensor,  # [N], on device
    input_ids_all,  # [N, L], on device
    attention_mask_all,  # [N, L], on device
    pairs,  # list of (winner_idx, loser_idx)
    epochs=10,
    lr=1e-5,
    beta=0.1,
    run_idx=0,
    desc_prefix="",
):
    """
    Much faster DPO training:
    - One forward over ALL completions per epoch
    - No tokenization or ref model forward in the loop
    """
    optimizer = torch.optim.AdamW(policy_model.parameters(), lr=lr)

    winners = torch.tensor([w for w, _ in pairs], dtype=torch.long, device=device)
    losers = torch.tensor([l for _, l in pairs], dtype=torch.long, device=device)

    log_probs_over_time = []

    use_amp = device == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    for epoch in range(epochs):
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=use_amp):
            # Forward once on all completions
            policy_log_probs = get_log_prob_sums_from_inputs(
                policy_model, input_ids_all, attention_mask_all
            )  # [N]

            # Gather for the pairs
            policy_w = policy_log_probs[winners]
            policy_l = policy_log_probs[losers]

            ref_w = ref_log_probs_tensor[winners]
            ref_l = ref_log_probs_tensor[losers]

            logits = beta * ((policy_w - ref_w) - (policy_l - ref_l))
            losses = -nn.functional.logsigmoid(logits)
            loss = losses.mean()

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Save per-example log-probs for metrics (just detach)
        log_probs_over_time.append(policy_log_probs.detach().cpu().numpy().tolist())

        if (epoch + 1) % 50 == 0:
            print(
                f"{desc_prefix} Run {run_idx + 1}: "
                f"Epoch {epoch + 1}/{epochs}, loss={loss.item():.4f}"
            )

    return {"log_probs": log_probs_over_time}


# ================================
# 7. Agreement metric
# ================================


def count_pair_agreements(model_scores, oracle_scores):
    """
    Count how many pairwise preferences (i,j) the model agrees on
    with the oracle ordering given by oracle_scores.

    model_scores: array-like of shape [N]
    oracle_scores: array-like of shape [N] (oracle ranking scores)
    """
    model_scores = np.array(model_scores)
    oracle_scores = np.array(oracle_scores)
    n = len(oracle_scores)
    assert len(model_scores) == n

    agreements = 0
    total = n * (n - 1) // 2

    for i in range(n):
        for j in range(i + 1, n):
            oracle_pref = oracle_scores[i] - oracle_scores[j]
            model_pref = model_scores[i] - model_scores[j]

            # Agreement if both prefer the same direction
            if oracle_pref * model_pref > 0:
                agreements += 1

    return agreements, total


# ================================
# 8. Multi-run experiment
# ================================

all_ref_frac = []
all_baseline_frac_over_time = []
all_graph_frac_over_time = []

for run_idx in range(NUM_RUNS):
    print(f"\n================ Run {run_idx + 1}/{NUM_RUNS} ================")

    # Different seed per run
    seed = BASE_SEED + run_idx
    torch.manual_seed(seed)
    np.random.seed(seed)

    # 2. Model Initialization (fresh models each run)
    pi_ref = GPT2LMHeadModel.from_pretrained(model_name)
    pi_ref.config.use_cache = False
    pi_ref.to(device)
    pi_ref.eval()

    pi_train_baseline = copy.deepcopy(pi_ref)
    pi_train_graph = copy.deepcopy(pi_ref)
    pi_train_baseline.train()
    pi_train_graph.train()

    print("Models initialized: pi_ref, pi_train_baseline, pi_train_graph")

    # 3. Data: completions for this run
    completions = generate_completions(
        pi_ref, tokenizer, prompt, n=N, max_length=20, run_idx=run_idx
    )
    print(f"Run {run_idx + 1}: Generated {len(completions)} completions.")

    # 3.1 Oracle ranking over completions (random permutation, hidden from stdout)
    perm = np.random.permutation(N)
    oracle_scores = np.empty(N)
    for rank, idx in enumerate(perm):
        oracle_scores[idx] = rank + 1  # ranks 1..N, higher = more preferred

    print(f"Run {run_idx + 1}: Oracle ranking generated.")

    # 4.1 Sample K oracle comparisons ("human feedback")
    sampled_indices = sample_pairs(N, K)
    labeled_sample = label_pairs(sampled_indices, oracle_scores)

    # 4.2 Estimate scores with Bradley–Terry from the sampled pairs
    estimated_scores = fit_bradley_terry(N, labeled_sample)

    # 4.2.1 Evaluate BT ranking accuracy vs oracle using the same pairwise metric
    bt_agree, total_pairs_bt = count_pair_agreements(estimated_scores, oracle_scores)
    bt_frac = bt_agree / total_pairs_bt
    print(
        f"Run {run_idx + 1}: BT-estimated ranking agreement: "
        f"{bt_agree} / {total_pairs_bt} ({bt_frac:.3f} fraction)"
    )

    # 4.3 Construct a full graph from estimated BT scores
    bt_full_graph_pairs = construct_full_graph(estimated_scores)

    print(
        f"Run {run_idx + 1}: Sampled {K} oracle pairs and constructed "
        f"BT-estimated full graph with {len(bt_full_graph_pairs)} edges."
    )

    # 5. Pre-tokenize all completions once and compute ref log-probs once
    tokens = tokenizer(completions, return_tensors="pt", padding=True, truncation=True)
    input_ids_all = tokens["input_ids"].to(device)
    attention_mask_all = tokens["attention_mask"].to(device)

    with torch.no_grad():
        ref_log_probs_tensor = get_log_prob_sums_from_inputs(
            pi_ref, input_ids_all, attention_mask_all
        )  # [N]

    ref_log_probs = ref_log_probs_tensor.detach().cpu().numpy().tolist()
    ref_log_probs_tensor = ref_log_probs_tensor.to(device)

    # 5.1 Train Baseline DPO (on sampled pairs only)
    print(f"Run {run_idx + 1}: Training Baseline DPO (sampled pairs)...")
    metrics_baseline = train_dpo_epoch_save_log_probs_fast(
        pi_train_baseline,
        ref_log_probs_tensor,
        input_ids_all,
        attention_mask_all,
        labeled_sample,  # K sampled oracle-labeled pairs
        epochs=epochs,
        lr=1e-5,
        beta=0.1,
        run_idx=run_idx,
        desc_prefix="Baseline",
    )

    # 5.2 Train Graph DPO (on BT-estimated full graph)
    print(f"Run {run_idx + 1}: Training Graph DPO (BT-estimated full graph)...")
    metrics_graph = train_dpo_epoch_save_log_probs_fast(
        pi_train_graph,
        ref_log_probs_tensor,
        input_ids_all,
        attention_mask_all,
        bt_full_graph_pairs,  # full graph induced by BT scores
        epochs=epochs,
        lr=1e-5,
        beta=0.1,
        run_idx=run_idx,
        desc_prefix="Graph",
    )

    # 6. Pairwise agreement metrics w.r.t. oracle ranking
    epochs_trained = len(metrics_baseline["log_probs"])

    # Agreement for the fixed reference policy (this run)
    ref_agree, total_pairs = count_pair_agreements(ref_log_probs, oracle_scores)
    ref_frac = ref_agree / total_pairs

    baseline_frac_over_time = []
    graph_frac_over_time = []

    for e in range(epochs_trained):
        baseline_scores_e = np.array(metrics_baseline["log_probs"][e])
        graph_scores_e = np.array(metrics_graph["log_probs"][e])

        baseline_agree_e, _ = count_pair_agreements(baseline_scores_e, oracle_scores)
        graph_agree_e, _ = count_pair_agreements(graph_scores_e, oracle_scores)

        baseline_frac_over_time.append(baseline_agree_e / total_pairs)
        graph_frac_over_time.append(graph_agree_e / total_pairs)

    print(f"Run {run_idx + 1}: Total possible pairs: {total_pairs}")
    print(
        f"Run {run_idx + 1}: pi_ref agreement (constant): "
        f"{ref_agree} / {total_pairs} ({ref_frac:.3f} fraction)"
    )
    print(
        f"Run {run_idx + 1}: Final agreement fractions - "
        f"Baseline DPO: {baseline_frac_over_time[-1]:.3f}, "
        f"Graph DPO: {graph_frac_over_time[-1]:.3f}"
    )

    # Store per-run results
    all_ref_frac.append(ref_frac)
    all_baseline_frac_over_time.append(baseline_frac_over_time)
    all_graph_frac_over_time.append(graph_frac_over_time)

    # Clear models to avoid memory issues
    del pi_ref, pi_train_baseline, pi_train_graph
    if device == "cuda":
        torch.cuda.empty_cache()

# ================================
# 9. Aggregate results over runs
# ================================

all_ref_frac = np.array(all_ref_frac)  # shape [NUM_RUNS]
all_baseline_frac_over_time = np.array(
    all_baseline_frac_over_time
)  # shape [NUM_RUNS, epochs]
all_graph_frac_over_time = np.array(
    all_graph_frac_over_time
)  # shape [NUM_RUNS, epochs]

avg_ref_frac = all_ref_frac.mean()
avg_baseline_frac_over_time = all_baseline_frac_over_time.mean(axis=0)
avg_graph_frac_over_time = all_graph_frac_over_time.mean(axis=0)

final_baseline_avg = avg_baseline_frac_over_time[-1]
final_graph_avg = avg_graph_frac_over_time[-1]

print("\n========== AVERAGED RESULTS OVER RUNS ==========")
print(f"Average pi_ref agreement fraction over {NUM_RUNS} runs: {avg_ref_frac:.3f}")
print(
    f"Average final agreement fraction - "
    f"Baseline DPO: {final_baseline_avg:.3f}, "
    f"Graph DPO: {final_graph_avg:.3f}"
)

# Variance of updates (epoch-to-epoch changes) for the averaged trajectories
baseline_deltas = np.diff(avg_baseline_frac_over_time[100:])
graph_deltas = np.diff(avg_graph_frac_over_time[100:])

baseline_var_updates = np.var(baseline_deltas)
graph_var_updates = np.var(graph_deltas)

print(
    "\nVariance of epoch-to-epoch agreement changes (averaged trajectories, after first 100 epochs):"
)
print(f"  DPO (sampled pairs): {baseline_var_updates:.6e}")
print(f"  Graph DPO (BT full graph): {graph_var_updates:.6e}")

# ================================
# 10. Visualization (Averaged)
# ================================

plt.figure(figsize=(8, 5))
x = list(range(epochs))

# Reference policy: constant line (average over runs)
plt.plot(
    x,
    [avg_ref_frac] * epochs,
    label="Ref",
    linestyle="--",
)

# Baseline and Graph DPO trajectories (averaged)
plt.plot(
    x,
    avg_baseline_frac_over_time,
    label="DPO",
    marker="o",
    markersize=4,
)
plt.plot(
    x,
    avg_graph_frac_over_time,
    label="GGDPO",
    marker="x",
    markersize=4,
)

plt.xlabel("Epochs")
plt.ylabel("Fraction of Pairwise Agreements w.r.t. Oracle Ranking")
plt.title(f"Pairwise Agreement with Oracle (Averaged over {NUM_RUNS} runs)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
